# Mining a Declare model and measuring fitness

This notebook mines a Declare model from a small event log and scores traces against it. The log, `loan_applications.csv`, holds eight cases of a loan application process, one row per event with the columns `case_id`, `activity`, and `timestamp`.

## Load the log

FCFDeclare reads the control flow only, so a log is a list of traces and a trace is the list of its activity names in order. The events of each case are sorted by timestamp, since the rows of a CSV need not be in order.

In [1]:
import csv
from collections import defaultdict

import fcfdeclare as fcf

events = defaultdict(list)
with open('loan_applications.csv', newline='') as file:
    for row in csv.DictReader(file):
        events[row['case_id']].append((row['timestamp'], row['activity']))

cases = {case: [activity for _, activity in sorted(rows)] for case, rows in events.items()}
log = list(cases.values())
for case, trace in cases.items():
    print(case, ' -> '.join(trace))

C1 Register -> Check -> Approve -> Notify -> Close
C2 Register -> Check -> Reject -> Notify -> Close
C3 Register -> Check -> Request Info -> Check -> Approve -> Notify -> Close
C4 Register -> Check -> Approve -> Notify -> Close
C5 Register -> Check -> Reject -> Notify -> Close
C6 Register -> Check -> Request Info -> Check -> Reject -> Notify -> Close
C7 Register -> Check -> Approve -> Notify -> Close
C8 Register -> Check -> Approve -> Close


## Mine the model

Every template is a candidate on every activity, or on every ordered pair of distinct activities. A candidate is kept when both its measures reach their thresholds:

- **support**: the fraction of the traces in which it is non-vacuously satisfied, that is, its activation occurs and it holds.
- **confidence**: the ratio between its support and the support of its activation, the fraction of the traces in which the activation occurs.

In [2]:
model = fcf.mine(log, min_support=0.5, min_confidence=0.85, max_cardinality=2)
print(f'{len(model.constraints)} constraints')

179 constraints


Each constraint maps to its measures, and the constraints are ordered by template, then by count, then by activity names. The first five include `Existence(Notify, 1)`, with a confidence under 1 because `C8` never notifies the applicant.

In [3]:
from itertools import islice

for constraint, measures in islice(model.constraints.items(), 5):
    print(f'{constraint}: support {measures.support:.2f}, confidence {measures.confidence:.2f}')

Existence(Check, 1): support 1.00, confidence 1.00
Existence(Close, 1): support 1.00, confidence 1.00
Existence(Notify, 1): support 0.88, confidence 0.88
Existence(Register, 1): support 1.00, confidence 1.00
Absence(Approve, 2): support 1.00, confidence 1.00


### Support and confidence on rare activities

Confidence also lets the miner keep informative rules on rare activities. `Request Info` occurs in only 2 of the 8 cases, so no constraint it activates can reach a support above 0.25. One of them is nonetheless a firm rule: whenever the applicant is asked for more information, the application is checked again right after. With `min_support=0.5`, it is not mined.

In [4]:
rule = fcf.Constraint('Chain Response', ('Request Info', 'Check'))
print(f'{rule} in the model: {rule in model.constraints}')

Chain Response(Request Info, Check) in the model: False


Lowering `min_support` to 0.2 and requiring a confidence of 1 recovers it, since no case breaks it. The support threshold still asks for an activation in at least 2 of the 8 cases, so no rule rests on a single case.

In [5]:
rare_model = fcf.mine(log, min_support=0.2, min_confidence=1.0, max_cardinality=2)
print(f'{len(rare_model.constraints)} constraints')
measures = rare_model.constraints[rule]
print(f'{rule}: support {measures.support:.2f}, confidence {measures.confidence:.2f}')

226 constraints
Chain Response(Request Info, Check): support 0.25, confidence 1.00


The rare rules show in conformance checking. A trace that approves right after requesting information breaks both models, since `Approve` no longer follows `Check` directly, but only the second one names the rule it breaks: a request for information is followed by a check.

In [6]:
no_recheck = [['Register', 'Check', 'Request Info', 'Approve', 'Notify', 'Close']]
for name, mined in [('model', model), ('rare_model', rare_model)]:
    row = fcf.Checker(mined).holds(no_recheck)[0]
    constraints = list(mined.constraints)
    print(f'{name}: {(~row).sum()} violated')
    for k in (~row).nonzero()[0]:
        print(f'    {constraints[k]}')

model: 2 violated
    Chain Precedence(Check, Approve)
    Not Chain Precedence(Request Info, Approve)
rare_model: 6 violated
    Response(Request Info, Check)
    Chain Response(Request Info, Check)
    Chain Precedence(Check, Approve)
    Not Chain Response(Request Info, Approve)
    Not Chain Precedence(Request Info, Approve)
    Alternate Response(Request Info, Check)


### Cardinality

`max_cardinality` sets the largest count $n$ for which Existence, Absence, and Exactly are candidates. Absence$(a, n)$ holds when $a$ occurs fewer than $n$ times, so Absence$(a, 2)$ says that $a$ occurs at most once, a rule that `max_cardinality=1` cannot express. The model mined above keeps it for every activity except `Check`, which `C3` and `C6` repeat after a request for information.

In [7]:
for constraint, measures in model.constraints.items():
    if constraint.n == 2:
        print(f'{constraint}: support {measures.support:.2f}, confidence {measures.confidence:.2f}')

Absence(Approve, 2): support 1.00, confidence 1.00
Absence(Close, 2): support 1.00, confidence 1.00
Absence(Notify, 2): support 1.00, confidence 1.00
Absence(Register, 2): support 1.00, confidence 1.00
Absence(Reject, 2): support 1.00, confidence 1.00
Absence(Request Info, 2): support 1.00, confidence 1.00


## Fitness of the log

A `Checker` scores a batch of traces against the model. A trace in which a constraint's activation does not occur satisfies it vacuously. Two scores come out for each trace:

- `share`: the fraction of the constraints it satisfies, a fitness in $[0, 1]$.
- `full`: 1 when it satisfies every constraint, else 0.

In [8]:
checker = fcf.Checker(model)
result = checker.check(log)
for case, share, full in zip(cases, result.share, result.full, strict=True):
    print(f'{case}: fitness {share:.3f}, conformant {bool(full)}')
print(f'Mean fitness {result.share.mean():.3f}, conformant cases {result.full.mean():.0%}')

C1: fitness 1.000, conformant True
C2: fitness 1.000, conformant True
C3: fitness 1.000, conformant True
C4: fitness 1.000, conformant True
C5: fitness 1.000, conformant True
C6: fitness 1.000, conformant True
C7: fitness 1.000, conformant True
C8: fitness 0.911, conformant False
Mean fitness 0.989, conformant cases 88%


## Fitness of new traces

The model also scores traces it was not mined from. Activities it does not name match no constraint, but still count as events.

In [9]:
new = {
    'conformant': ['Register', 'Check', 'Reject', 'Notify', 'Close'],
    'skips check': ['Register', 'Approve', 'Notify', 'Close'],
    'approve and reject': ['Register', 'Check', 'Approve', 'Reject', 'Notify', 'Close'],
    'unknown activity': ['Register', 'Check', 'Escalate', 'Approve', 'Notify', 'Close'],
}
result = checker.check(new.values())
for name, share in zip(new, result.share, strict=True):
    print(f'{name}: fitness {share:.3f}')

conformant: fitness 1.000
skips check: fitness 0.883
approve and reject: fitness 0.972
unknown activity: fitness 0.994


`holds` gives the satisfaction of each constraint separately, one column per constraint of the model, which shows which rules a trace breaks.

In [10]:
holds = checker.holds(new.values())
constraints = list(model.constraints)
for name, row in zip(new, holds, strict=True):
    violated = [constraints[k] for k in (~row).nonzero()[0]]
    print(f'{name}: {len(violated)} violated')
    for constraint in violated:
        print(f'    {constraint}')

conformant: 0 violated
skips check: 21 violated
    Existence(Check, 1)
    Choice(Check, Reject)
    Choice(Check, Request Info)
    Choice(Reject, Check)
    Choice(Request Info, Check)
    Responded Existence(Approve, Check)
    Responded Existence(Close, Check)
    Responded Existence(Notify, Check)
    Responded Existence(Register, Check)
    Response(Register, Check)
    Precedence(Check, Approve)
    Precedence(Check, Close)
    Precedence(Check, Notify)
    Chain Response(Register, Check)
    Chain Precedence(Check, Approve)
    Not Chain Response(Register, Approve)
    Not Chain Precedence(Register, Approve)
    Alternate Response(Register, Check)
    Alternate Precedence(Check, Approve)
    Alternate Precedence(Check, Close)
    Alternate Precedence(Check, Notify)
approve and reject: 5 violated
    Exclusive Choice(Approve, Reject)
    Exclusive Choice(Reject, Approve)
    Not Responded Existence(Approve, Reject)
    Not Response(Approve, Reject)
    Not Chain Response(Approv

## Save and reload

A model saves to JSON with its settings and measures, and reloads to an equal model.

In [11]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'model.json'
    model.save(path)
    reloaded = fcf.DeclareModel.load(path)
reloaded == model

True